# 🚗 Vehicle Fuel Efficiency Prediction
## Notebook 4: Exploratory Data Analysis (EDA)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
PALETTE = sns.color_palette('husl', 8)

df = pd.read_csv('../data/auto-mpg-cleaned.csv')
print(f'Dataset: {df.shape}')
df.head()

## 4.1 Univariate Analysis — All Features

In [ ]:
numeric_cols = ['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=28, color=PALETTE[i],
                 edgecolor='white', linewidth=0.4, alpha=0.85)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--',
                    linewidth=1.5, label=f'Mean: {df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='orange', linestyle=':',
                    linewidth=1.5, label=f'Median: {df[col].median():.1f}')
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].legend(fontsize=9)

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Univariate Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/03_univariate_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.2 Bivariate Analysis — Features vs MPG

In [ ]:
features = ['displacement', 'horsepower', 'weight', 'acceleration']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].scatter(df[feat], df['mpg'], alpha=0.5, color=PALETTE[i+1], s=40, edgecolors='white', linewidths=0.3)

    # Trend line
    z = np.polyfit(df[feat].dropna(), df.loc[df[feat].notna(), 'mpg'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 200)
    axes[i].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')

    corr = df[[feat, 'mpg']].corr().iloc[0, 1]
    axes[i].set_title(f'{feat} vs MPG  (r = {corr:.3f})', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('MPG')
    axes[i].legend()

plt.suptitle('Feature vs Target (MPG)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/04_bivariate_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.3 Categorical Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Cylinders vs MPG
df.boxplot(column='mpg', by='cylinders', ax=axes[0],
           patch_artist=True, notch=False)
axes[0].set_title('MPG by Cylinders', fontweight='bold')
axes[0].set_xlabel('Cylinders')
axes[0].set_ylabel('MPG')

# Origin vs MPG
origin_labels = {1: 'USA', 2: 'Europe', 3: 'Japan'}
df_origin = df.copy()
df_origin['origin_label'] = df_origin['origin'].map(origin_labels)
df_origin.boxplot(column='mpg', by='origin_label', ax=axes[1],
                  patch_artist=True)
axes[1].set_title('MPG by Origin', fontweight='bold')
axes[1].set_xlabel('Origin')
axes[1].set_ylabel('MPG')

# Model Year vs Avg MPG
year_mpg = df.groupby('model_year')['mpg'].mean().reset_index()
axes[2].bar(year_mpg['model_year'] + 1900, year_mpg['mpg'],
            color='steelblue', edgecolor='white', linewidth=0.4)
axes[2].set_title('Avg MPG by Model Year', fontweight='bold')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Avg MPG')
axes[2].xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))

plt.suptitle('', fontsize=1)
plt.tight_layout()
plt.savefig('../plots/05_categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Correlation Heatmap

In [ ]:
numeric_df = df[['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']]
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, square=True, annot_kws={'size': 11})
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../plots/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation with MPG (sorted):')
print(corr_matrix['mpg'].drop('mpg').sort_values(ascending=False))

## 4.5 Pair Plot

In [ ]:
pair_cols = ['mpg', 'displacement', 'horsepower', 'weight', 'acceleration']
g = sns.pairplot(df[pair_cols], diag_kind='kde', plot_kws={'alpha': 0.4, 's': 25},
                 diag_kws={'fill': True, 'color': 'steelblue'})
g.figure.suptitle('Pair Plot of Key Features', y=1.02, fontsize=14, fontweight='bold')
g.savefig('../plots/07_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

## 4.6 Multicollinearity Check (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = numeric_df.drop(columns=['mpg']).dropna()
vif_df = pd.DataFrame({
    'Feature': vif_data.columns,
    'VIF': [variance_inflation_factor(vif_data.values, i) for i in range(vif_data.shape[1])]
}).sort_values('VIF', ascending=False)

print('Variance Inflation Factor (VIF):')
print('VIF > 10 → High multicollinearity')
display(vif_df)

# Plot VIF
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(vif_df['Feature'], vif_df['VIF'],
               color=['#e74c3c' if v > 10 else '#3498db' for v in vif_df['VIF']])
ax.axvline(10, color='red', linestyle='--', linewidth=1.5, label='VIF = 10 (threshold)')
ax.set_title('Variance Inflation Factor (VIF)', fontsize=13, fontweight='bold')
ax.set_xlabel('VIF Score')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/08_vif.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Key EDA Findings

| Finding | Implication |
|---------|-------------|
| Weight, displacement, horsepower are highly negatively correlated with MPG | Heavier, larger-engine cars = less efficient |
| Acceleration is positively correlated with MPG | Lighter, efficient cars also accelerate quicker |
| 4-cylinder cars have far higher MPG than 8-cylinder | Cylinder count is key predictor |
| Japan/European cars outperform US cars in MPG | Origin is a meaningful feature |
| MPG improved significantly from 1970 → 1982 | Fuel crisis effect (1973 OPEC shock) |
| High VIF for displacement, cylinders, weight | Multicollinearity — tree models handle this better than linear |